# ADC Example 1 — Single vs. Continuous Conversion

**Target:** STM32F103C8T6 · ADC1 · Channel 0 (PA0)
**Focus:** The conversion *rhythm* — one-shot vs. free-running.

---
## CubeMX Configuration

### Common clock setup
- **HCLK** = 72 MHz.
- **ADC Prescaler** = 6 → **ADCCLK = 12 MHz** (must stay ≤ 14 MHz).

### Mode A — Single Conversion
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Disabled |
| `Continuous Conversion Mode` | **Disabled** |
| `Number Of Conversion` | 1 |
| Rank 1 | Channel 0, Sampling Time 55.5 Cycles |

With continuous mode **disabled**, the ADC performs one conversion per `HAL_ADC_Start()` call and then stops. The `DR` register holds the last result until you read it or start another conversion.

### Mode B — Continuous Conversion
| Parameter | Setting |
| :--- | :--- |
| `Scan Conversion Mode` | Disabled |
| `Continuous Conversion Mode` | **Enabled** |
| `Number Of Conversion` | 1 |
| Rank 1 | Channel 0, Sampling Time 55.5 Cycles |

With continuous mode **enabled**, the ADC restarts automatically after every conversion. You call `HAL_ADC_Start()` **once**, and the ADC free-runs until `HAL_ADC_Stop()` is called[reference:0].

> ⚠️ **Continuous mode requires DMA or interrupt.** If you only poll `HAL_ADC_GetValue()`, you will likely miss conversions because `DR` is overwritten at the ADC clock rate.

## HAL Initialization (generated by CubeMX)

The relevant fields in `MX_ADC1_Init()` differ only in one line:

```c
// Mode A — Single
hadc1.Init.ContinuousConvMode = DISABLE;

// Mode B — Continuous
hadc1.Init.ContinuousConvMode = ENABLE;
```

All other fields (scan disabled, one rank, channel 0) remain the same.

## User Code — Single Conversion (Polling)

Place this inside `while (1)` in `main.c`, between `USER CODE BEGIN WHILE` and `USER CODE END WHILE`.

In [ ]:
uint32_t adc_raw;
float voltage;

HAL_ADC_Start(&hadc1);                         // start ONE conversion
HAL_ADC_PollForConversion(&hadc1, 10);         // wait up to 10 ms
if (HAL_ADC_GetState(&hadc1) & HAL_ADC_STATE_EOC) {
    adc_raw   = HAL_ADC_GetValue(&hadc1);
    voltage   = (float)adc_raw * 3.3f / 4095.0f;
}

HAL_Delay(100);

## User Code — Continuous Conversion + Interrupt

For continuous mode, use the interrupt callback instead of polling. Start the ADC once; the callback fires repeatedly at the conversion rate.

In [ ]:
volatile uint32_t adc_raw;
volatile uint8_t  adc_fresh = 0;

// In main(), before while(1):
HAL_ADC_Start_IT(&hadc1);   // start continuous conversion + EOC interrupt

// Callback (outside main):
void HAL_ADC_ConvCpltCallback(ADC_HandleTypeDef *hadc) {
    if (hadc->Instance == ADC1) {
        adc_raw   = HAL_ADC_GetValue(hadc);
        adc_fresh = 1;
    }
}

// In while(1):
if (adc_fresh) {
    adc_fresh = 0;
    float voltage = (float)adc_raw * 3.3f / 4095.0f;
    // use voltage
}

## Behaviour Summary

| | Single | Continuous |
| :--- | :--- | :--- |
| `HAL_ADC_Start` calls needed | One per sample | **One total** |
| `DR` update rate | Only after each start | ADC clock rate / conversion time |
| Suitable access method | Polling, IT, DMA | IT or DMA (DMA strongly preferred) |
| Typical use | Button-triggered measurement, low-power | Real-time monitoring, audio, control loops |